In [1]:
# -*- coding: utf-8 -*-
"""
复现：华泰期货《国债期货和国债ETF的统计套利策略（上篇+下篇）》
运行环境：聚宽研究环境（Python 3.6 / numpy 1.19 / pandas 0.23 / scipy 1.2）
特点：
  * 所有数据直接用 get_price / get_dominant_future 提取，不写磁盘；结果全部 print
  * 不依赖 pywt / hmmlearn：小波(Haar 平稳小波，因果版)与高斯HMM均为 numpy 手写
  * 全流程因果（无未来函数）：卡尔曼滤波、小波、DDIVF、HMM 状态均为"滚动/递推"计算
流程：
  30分钟K线 -> TL主力合约(换月价差平滑) -> 流动性因子(有效价差) -> 卡尔曼滤波(时变对冲比率+新息)
  -> 小波取高频新息 -> DDIVF 波动率 -> z 值 -> HMM 两状态(5维观测)
  -> 滚动窗口网格遍历 P0/P1 使训练集夏普最大(训练集亏损则空仓) -> 逐K线回测(含滑点/手续费)
"""
import datetime
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
try:
    from jqdata import *          # 聚宽研究环境
except Exception:
    pass

# =============================== 参数区 ===============================
START_DATE   = '2024-01-01'       # 回测/统计起点
END_DATE     = '2026-12-20'       # 回测终点（超过今天自动截断）
WARMUP_START = '2023-08-01'       # 预热数据起点（卡尔曼初始化、HMM训练窗口需要历史；研报511090建模起点为2023/08）
ETF_LIST = [('511130.XSHG', '博时上证30年期国债ETF'),
            ('511090.XSHG', '鹏扬中证30年期国债ETF')]
TL_SYMBOL = 'TL'

BARS_PER_DAY   = 8                # 30分钟K线：10:00,10:30,11:00,11:30,13:30,14:00,14:30,15:00
BARS_PER_YEAR  = 252 * BARS_PER_DAY

# --- 模型参数 ---
INIT_DAYS     = 30                # 卡尔曼初始化(OLS)所用交易日数
LIQ_WIN       = 20                # 有效价差(Roll)滚动窗口(K线数)
LIQ_NORM_WIN  = 480               # 流动性因子归一化(滚动z-score)窗口
Q_RATIO       = 1e-5              # 卡尔曼状态噪声 Q = Q_RATIO * R * I
WAVELET_LEVEL = 4                 # 小波分解层数（研报图1为4层）
ALPHA_GRID    = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]   # DDIVF平滑系数候选
HMM_STATES    = 2
HMM_ITER      = 40

# --- 滚动参数遍历 ---
TRAIN_DAYS = 60                   # 训练窗口(交易日)
UPD_DAYS   = 20                   # 参数更新间隔(交易日)
PCT_GRID   = [1, 2, 3, 5, 7, 10, 15, 20, 30]   # P0/P1 候选：该状态下 z 的分位数(%)，z低于该分位数则做多价差
EXIT_Z     = 0.0                  # z 回到 >= EXIT_Z 平仓

# --- 回测参数（与研报一致） ---
INIT_CAPITAL   = 1e7
ETF_WEIGHT     = 0.9              # 90%资金用于ETF，10%为期货保证金
ETF_TICK       = 0.001
ETF_SLIP_TICKS = 3                # ETF 滑点 3 跳
ETF_FEE        = 0.85e-4          # ETF 手续费 万0.85
TL_TICK        = 0.01
TL_SLIP_TICKS  = 1                # TL 滑点 1 跳
TL_FEE_YUAN    = 3.0              # TL 手续费 3元/手
TL_MULT        = 10000.0          # TL 合约乘数：面值100万 -> 报价*10000

# --- 研报上篇的"累计收益率之差"策略 ---
RUN_PART1 = True
PART1_X   = 5                     # 观察窗口(天)
PART1_K   = 0.1

SHOW_PLOT = False                 # 需要图时改 True（plt.show 内联显示，不写磁盘）


# =============================== 工具函数 ===============================
def _d(s):
    return datetime.datetime.strptime(s, '%Y-%m-%d').date()


def get_bars30(code, d0, d1):
    """取 [d0, d1] 含端点的30分钟K线(open/close)；分钟数据 end_date 需带时间，否则不含当天"""
    sdt = datetime.datetime.combine(d0, datetime.time(0, 0))
    edt = datetime.datetime.combine(d1, datetime.time(16, 0))
    df = get_price(code, start_date=sdt, end_date=edt, frequency='30m',
                   fields=['open', 'close'], skip_paused=True)
    if df is None:
        return pd.DataFrame(columns=['open', 'close'])
    return df.dropna()


def _last_close_on(df, d):
    sub = df[df.index.date == d]
    if len(sub) == 0:
        return None
    return float(sub['close'].iloc[-1])

def get_cal_days(d0, d1):
    """交易日历: 先试 get_trade_days, 否则用 TL9999 日线日期"""
    try:
        return [pd.Timestamp(x).date() for x in get_trade_days(start_date=d0, end_date=d1)]
    except Exception:
        pass
    try:
        from jqdata import get_trade_days as _g
        return [pd.Timestamp(x).date() for x in _g(start_date=d0, end_date=d1)]
    except Exception:
        pass
    df = get_price('TL9999.CCFX', start_date=d0, end_date=d1, frequency='daily',
                   fields=['close'], skip_paused=True)
    return [pd.Timestamp(x).date() for x in df.index]

def build_tl(d0, d1):
    """
    构造 TL 主力连续30分钟序列：逐日取主力合约，换月处做加法平移(后复权到最新合约价格水平)，
    保证价差序列与盈亏连续。返回列：of_/cf(平滑后开/收), ofr/cfr(原始开/收，用于下单手数)
    """
    days = get_cal_days(d0, d1)
    dom = []
    for d in days:
        try:
            c = get_dominant_future(TL_SYMBOL, d)
        except Exception:
            c = None
        dom.append(c)
    seg = []
    for i, c in enumerate(dom):
        if not c:
            continue
        if seg and seg[-1][0] == c:
            seg[-1][2] = i
        else:
            seg.append([c, i, i])
    raw = {}
    for c, a, b in seg:
        raw[c] = get_bars30(c, days[max(a - 1, 0)], days[b])
    cum = [0.0] * len(seg)
    for k in range(len(seg) - 1, 0, -1):
        c_old, c_new = seg[k - 1][0], seg[k][0]
        d_prev = days[seg[k][1] - 1]
        co, cn = _last_close_on(raw[c_old], d_prev), _last_close_on(raw[c_new], d_prev)
        off = (cn - co) if (co is not None and cn is not None) else 0.0
        cum[k - 1] = cum[k] + off
    parts = []
    for k, (c, a, b) in enumerate(seg):
        df = raw[c]
        ds = set(days[a:b + 1])
        m = np.array([x in ds for x in df.index.date])
        sub = df[m]
        out = pd.DataFrame({'of_': sub['open'].values + cum[k], 'cf': sub['close'].values + cum[k],
                            'ofr': sub['open'].values, 'cfr': sub['close'].values}, index=sub.index)
        parts.append(out)
    tl = pd.concat(parts)
    tl = tl[~tl.index.duplicated(keep='last')].sort_index()
    return tl


# =============================== 流动性因子 / 卡尔曼 / 小波 / DDIVF ===============================
def liq_factor(price, day_id):
    """有效价差(Roll): L = 2*sqrt(-Cov(dS_t, dS_{t-1})) (Cov<=0) 否则0；日内差分；再滚动z-score归一化"""
    s = pd.Series(np.asarray(price, float))
    d = s.groupby(day_id).diff()
    d1 = d.shift(1)
    mp = max(LIQ_WIN // 2, 5)
    cov = (d * d1).rolling(LIQ_WIN, min_periods=mp).mean() - \
          d.rolling(LIQ_WIN, min_periods=mp).mean() * d1.rolling(LIQ_WIN, min_periods=mp).mean()
    L = pd.Series(2 * np.sqrt(np.maximum(-cov.values, 0.0)))
    mu = L.rolling(LIQ_NORM_WIN, min_periods=40).mean()
    sd = L.rolling(LIQ_NORM_WIN, min_periods=40).std()
    z = ((L - mu) / sd.replace(0, np.nan)).values
    return np.clip(np.nan_to_num(z), -5, 5)


def kalman_filter(y, H, x0, P0, Q, R):
    """
    状态方程 x_t = x_{t-1} + w_t ; 测量方程 y_t = H_t x_t + u_t
    返回后验状态序列 X、新息 V (V_t = y_t - H_t x_{t|t-1})
    """
    T, n = H.shape
    X = np.zeros((T, n))
    V = np.zeros(T)
    x = x0.copy()
    P = P0.copy()
    for t in range(T):
        P = P + Q
        h = H[t]
        Ph = P.dot(h)
        S = h.dot(Ph) + R
        v = y[t] - h.dot(x)
        K = Ph / S
        x = x + K * v
        P = P - np.outer(K, Ph)
        P = 0.5 * (P + P.T)
        X[t] = x
        V[t] = v
    return X, V


def haar_swt_causal(v, level):
    """因果 Haar 平稳小波(à trous)：a_j[t]=(a_{j-1}[t]+a_{j-1}[t-2^{j-1}])/2, d_j=a_{j-1}-a_j"""
    a_prev = np.asarray(v, float).copy()
    details = []
    for j in range(level):
        s = 2 ** j
        sh = a_prev.copy()
        sh[s:] = a_prev[:-s]
        a = 0.5 * (a_prev + sh)
        details.append(a_prev - a)
        a_prev = a
    return a_prev, details


def ddivf(v, alpha, mp=30):
    """
    数据驱动新息波动率预测：sigma_t=(1-a)sigma_{t-1}+a*|v_{t-1}-vbar|/rho, rho=Corr(v-vbar, sign(v-vbar))
    vbar、rho 均用扩展窗口(因果)
    """
    s = pd.Series(v)
    x = v - s.expanding(min_periods=1).mean().values
    sg = np.sign(x)
    rho = pd.Series(x).expanding(min_periods=mp).corr(pd.Series(sg)).values
    rho = np.where(np.isnan(rho) | (rho <= 0.05), 0.7979, rho)
    target = np.abs(x) / rho
    N = len(v)
    sigma = np.zeros(N)
    sigma[0] = max(target[:mp].mean(), 1e-9)
    for t in range(1, N):
        sigma[t] = (1 - alpha) * sigma[t - 1] + alpha * target[t - 1]
    sigma = np.maximum(sigma, 1e-9)
    return x / sigma, sigma, target


# =============================== 高斯 HMM（对角协方差，numpy 实现） ===============================
def _log_gauss(X, means, vars_):
    T = X.shape[0]
    K = means.shape[0]
    out = np.zeros((T, K))
    for k in range(K):
        out[:, k] = -0.5 * (np.sum(np.log(2 * np.pi * vars_[k])) + np.sum((X - means[k]) ** 2 / vars_[k], axis=1))
    return out


def _hmm_forward(logB, A, pi):
    T, K = logB.shape
    B = np.exp(logB - logB.max(axis=1, keepdims=True))
    alpha = np.zeros((T, K))
    c = np.zeros(T)
    a = pi * B[0]
    c[0] = a.sum() + 1e-300
    alpha[0] = a / c[0]
    for t in range(1, T):
        a = alpha[t - 1].dot(A) * B[t]
        c[t] = a.sum() + 1e-300
        alpha[t] = a / c[t]
    return alpha, c, B


def fit_hmm(X, K=2, n_iter=40):
    """Baum-Welch；返回按"总方差升序"排序后的参数：状态0=平稳，状态1=动荡"""
    T, D = X.shape
    mag = np.abs(X).sum(axis=1)
    thr = np.percentile(mag, 70)
    gamma = np.zeros((T, K))
    gamma[mag <= thr, 0] = 1.0
    gamma[mag > thr, 1:] = 1.0 / max(K - 1, 1)
    Nk = gamma.sum(0) + 1e-8
    means = gamma.T.dot(X) / Nk[:, None]
    vars_ = np.maximum(gamma.T.dot(X ** 2) / Nk[:, None] - means ** 2, 1e-3)
    A = np.full((K, K), 0.1 / max(K - 1, 1))
    np.fill_diagonal(A, 0.9)
    pi = np.full(K, 1.0 / K)
    last_ll = -np.inf
    for _ in range(n_iter):
        logB = _log_gauss(X, means, vars_)
        alpha, c, B = _hmm_forward(logB, A, pi)
        beta = np.ones((T, K))
        for t in range(T - 2, -1, -1):
            beta[t] = A.dot(B[t + 1] * beta[t + 1]) / c[t + 1]
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
        xi = A * alpha[:-1].T.dot(B[1:] * beta[1:] / c[1:, None])
        pi = gamma[0] + 1e-6
        pi /= pi.sum()
        A = xi + 1e-6
        A /= A.sum(axis=1, keepdims=True)
        Nk = gamma.sum(0) + 1e-8
        means = gamma.T.dot(X) / Nk[:, None]
        vars_ = np.maximum(gamma.T.dot(X ** 2) / Nk[:, None] - means ** 2, 1e-3)
        ll = np.sum(np.log(c)) + np.sum(logB.max(axis=1))
        if abs(ll - last_ll) < 1e-4 * abs(ll):
            break
        last_ll = ll
    order = np.argsort(vars_.sum(axis=1))
    return {'A': A[order][:, order], 'pi': pi[order], 'means': means[order], 'vars': vars_[order]}


def hmm_filter(X, p):
    """前向滤波（只用当前及过去信息）：返回各时刻状态后验概率"""
    logB = _log_gauss(X, p['means'], p['vars'])
    alpha, _, _ = _hmm_forward(logB, p['A'], p['pi'])
    return alpha


# =============================== 数据准备与建模 ===============================
def build_model(df, verbose=True):
    N = len(df)
    dates = np.array(df.index.date)
    day_id = pd.factorize(dates)[0]
    day_start = np.r_[0, np.where(np.diff(day_id) != 0)[0] + 1]
    first = np.zeros(N, bool)
    first[day_start] = True
    last = np.r_[first[1:], True]
    m = {'N': N, 'dates': dates, 'day_id': day_id, 'day_start': day_start, 'first': first, 'last': last}
    for k in ['oe', 'ce', 'of_', 'cf', 'ofr', 'cfr']:
        m[k] = df[k].values.astype(float)
    ce, cf = m['ce'], m['cf']
    n0 = day_start[INIT_DAYS] if len(day_start) > INIT_DAYS else N // 4
    lnE, lnF = np.log(ce), np.log(cf)
    Le = liq_factor(ce, day_id)
    Lf = liq_factor(cf, day_id)
    H = np.column_stack([np.ones(N), lnF - lnF[:n0].mean(), Le, Lf])   # 对 lnP_TL 去中心化，仅改变alpha定义
    # ---- 研报表3：有/无流动性因子的回归 R^2 与 ADF p 值 ----
    if verbose:
        rows = []
        for nm, cols in [('无流动性因子', [0, 1]), ('有流动性因子', [0, 1, 2, 3])]:
            Xm = H[:, cols]
            b = np.linalg.lstsq(Xm, lnE, rcond=None)[0]
            res = lnE - Xm.dot(b)
            r2 = 1 - res.var() / lnE.var()
            try:
                from statsmodels.tsa.stattools import adfuller
                p = adfuller(res, autolag='AIC')[1]
            except Exception:
                p = np.nan
            rows.append([nm, '%.3f' % r2, '%.3f' % p])
        print('\n[表3 复现] 流动性因子对回归分析的影响(全样本OLS, 30分钟)')
        print(pd.DataFrame(rows, columns=['方程', 'R2', 'ADF检验P值']).to_string(index=False))
    # ---- 卡尔曼滤波 ----
    b0 = np.linalg.lstsq(H[:n0], lnE[:n0], rcond=None)[0]
    res0 = lnE[:n0] - H[:n0].dot(b0)
    R = max(res0.var(), 1e-10)
    P0 = R * np.linalg.pinv(H[:n0].T.dot(H[:n0]))
    Q = np.eye(4) * Q_RATIO * R
    P0 = P0 + Q
    Xs, V = kalman_filter(lnE, H, b0, P0, Q, R)
    beta = Xs[:, 1]
    if verbose:
        print('\n[卡尔曼] 对冲比率 beta1: 最小 %.4f  最大 %.4f  均值 %.4f | 新息均值绝对值 %.5f' %
              (beta[n0:].min(), beta[n0:].max(), beta[n0:].mean(), np.abs(V[n0:]).mean()))
    # ---- 小波：取高频(原序列 - L层近似) ----
    a4, dets = haar_swt_causal(V, WAVELET_LEVEL)
    vh = V - a4
    if verbose:
        comps = [a4] + dets[::-1]
        names = ['Approx(L%d)' % WAVELET_LEVEL] + ['Detail L%d' % i for i in range(WAVELET_LEVEL, 0, -1)]
        en = np.array([np.sum(c_[n0:] ** 2) for c_ in comps])
        en = en / en.sum()
        print('[小波] Haar 平稳小波能量占比(近似, 研报图1: A4 0.034 / D4 0.052 / D3 0.064 / D2 0.237 / D1 0.614):')
        print('   ' + ' | '.join('%s %.3f' % (nm, e) for nm, e in zip(names, en)))
    # ---- DDIVF：用首个训练节点之前的数据选 alpha，再递推得到 z ----
    d0 = INIT_DAYS + TRAIN_DAYS
    a_end = day_start[d0] if len(day_start) > d0 else N
    best, best_err = ALPHA_GRID[0], np.inf
    for al in ALPHA_GRID:
        _, sg, tg = ddivf(vh, al)
        err = np.mean((tg[n0:a_end] - sg[n0:a_end]) ** 2) if a_end > n0 else np.inf
        if err < best_err:
            best, best_err = al, err
    z, sigma, _ = ddivf(vh, best)
    z[:n0] = 0.0
    if verbose:
        print('[DDIVF] 最优平滑系数 alpha = %.2f' % best)
    # ---- 衍生数组 ----
    oe, of_, cfr = m['oe'], m['of_'], m['cfr']
    gap_e = np.r_[0.0, oe[1:] / ce[:-1] - 1]
    gap_f = np.r_[0.0, of_[1:] / cf[:-1] - 1]
    oc_e = ce / oe - 1
    oc_f = cf / of_ - 1
    r_e = np.r_[0.0, np.diff(np.log(ce))]
    r_f = np.r_[0.0, np.diff(np.log(cf))]
    m.update({'beta': beta, 'v': V, 'vh': vh, 'z': z, 'Le': Le, 'Lf': Lf, 'n0': n0,
              'gap_e': gap_e, 'gap_f': gap_f, 'oc_e': oc_e, 'oc_f': oc_f,
              'feat': np.column_stack([r_e, r_f, Le, Lf, vh])})
    return m


def build_nodes(m):
    """每 UPD_DAYS 个交易日一个节点：训练HMM(训练窗口) -> 前向滤波得到训练+测试区间的状态"""
    ds = m['day_start']
    n_days = len(ds)
    ds_ext = np.r_[ds, m['N']]
    nodes = []
    d0 = INIT_DAYS + TRAIN_DAYS
    for dk in range(d0, n_days, UPD_DAYS):
        a = ds_ext[dk - TRAIN_DAYS]
        b = ds_ext[dk]
        c = ds_ext[min(dk + UPD_DAYS, n_days)]
        F = m['feat']
        mu = F[a:b].mean(axis=0)
        sd = F[a:b].std(axis=0) + 1e-12
        par = fit_hmm((F[a:b] - mu) / sd, HMM_STATES, HMM_ITER)
        pr = hmm_filter((F[a:c] - mu) / sd, par)
        nodes.append({'tr_a': a, 'tr_b': b, 'te_b': c, 'st': np.argmax(pr, axis=1)})
    return nodes


# =============================== 训练集向量化模拟 / 逐K线回测 ===============================
def train_eval(z, st, beta, gap_e, gap_f, oc_e, oc_f, last, cost, th0, th1):
    """
    对 C 个 (P0,P1) 组合并行模拟(单位ETF名义本金的收益率)。
    bar i 收盘出信号 -> bar i+1 开盘成交(若 bar i 为当日最后一根，则不调仓)；隔夜缺口由旧仓位承担。
    """
    n = len(z)
    C = len(th0)
    pos = np.zeros(C)
    rets = np.zeros((n, C))
    for i in range(n - 1):
        if last[i]:
            new = pos
        else:
            thr = th0 if st[i] == 0 else th1
            enter = (pos == 0) & (z[i] < thr)
            leave = (pos == 1) & (z[i] >= EXIT_Z)
            new = np.where(enter, 1.0, np.where(leave, 0.0, pos))
        b = beta[i]
        rets[i + 1] = pos * (gap_e[i + 1] - b * gap_f[i + 1]) + new * (oc_e[i + 1] - b * oc_f[i + 1]) \
            - cost[i + 1] * np.abs(new - pos)
        pos = new
    return rets


def cost_array(m, slip_on, fee_on):
    oe, cfr, beta = m['oe'], m['cfr'], m['beta']
    c = slip_on * (ETF_SLIP_TICKS * ETF_TICK / oe + beta * TL_SLIP_TICKS * TL_TICK / cfr) \
        + fee_on * (ETF_FEE + beta * TL_FEE_YUAN / (cfr * TL_MULT))
    return c


def run_scenario(m, nodes, slip_on, fee_on, bt_i):
    N = m['N']
    z, beta = m['z'], m['beta']
    thr = np.zeros(N)
    active = np.zeros(N, bool)
    cost = cost_array(m, slip_on, fee_on)
    G = len(PCT_GRID)
    chosen = []
    for nd in nodes:
        a, b, c = nd['tr_a'], nd['tr_b'], nd['te_b']
        st = nd['st']
        ntr = b - a
        ztr, sttr = z[a:b], st[:ntr]
        qs = []
        for s in range(HMM_STATES):
            arr = ztr[sttr == s]
            if len(arr) < 30:
                arr = ztr
            qs.append(np.percentile(arr, PCT_GRID))
        th0 = np.repeat(qs[0], G)
        th1 = np.tile(qs[1], G)
        rets = train_eval(ztr, sttr, beta[a:b], m['gap_e'][a:b], m['gap_f'][a:b],
                          m['oc_e'][a:b], m['oc_f'][a:b], m['last'][a:b], cost[a:b], th0, th1)
        tot = rets.sum(axis=0)
        sd = rets.std(axis=0)
        sh = np.where(sd > 0, rets.mean(axis=0) / np.where(sd > 0, sd, 1.0) * np.sqrt(BARS_PER_YEAR), -np.inf)
        sh[tot <= 0] = -np.inf
        k = int(np.argmax(sh))
        ok = bool(np.isfinite(sh[k]))
        thr[b:c] = np.where(st[ntr:] == 0, th0[k], th1[k])
        active[b:c] = ok
        chosen.append((m['dates'][b], PCT_GRID[k // G], PCT_GRID[k % G], ok, sh[k] if ok else np.nan))
    res = run_backtest(m, thr, active, slip_on, fee_on, bt_i)
    res['chosen'] = chosen
    return res


def run_backtest(m, thr, active, slip_on, fee_on, bt_i):
    N = m['N']
    oe, ce, of_, cf, cfr = m['oe'], m['ce'], m['of_'], m['cf'], m['cfr']
    z, beta, first = m['z'], m['beta'], m['first']
    eq = INIT_CAPITAL
    pe, pf = 0.0, 0.0            # ETF份额(>=0)，期货手数(<=0 表示空头)
    curve = np.full(N, np.nan)
    pnls = []
    entry_eq = None
    slip_tot = fee_tot = 0.0
    for j in range(bt_i, N):
        if j > bt_i:
            eq += pe * (oe[j] - ce[j - 1]) + pf * TL_MULT * (of_[j] - cf[j - 1])     # 隔夜/跳空
            if not first[j]:                                                           # 开盘不调仓
                i = j - 1
                if not active[i]:
                    want = 0
                elif pe == 0:
                    want = 1 if z[i] < thr[i] else 0
                else:
                    want = 0 if z[i] >= EXIT_Z else 1
                if want == 1 and pe == 0:
                    b = beta[i]
                    lots = int(b * ETF_WEIGHT * eq / (cfr[j - 1] * TL_MULT))
                    if lots >= 1:
                        nf = lots * cfr[j - 1] * TL_MULT
                        sh = int(nf / b / oe[j] / 100.0) * 100
                        if sh > 0:
                            entry_eq = eq
                            s_c = sh * ETF_SLIP_TICKS * ETF_TICK + lots * TL_SLIP_TICKS * TL_TICK * TL_MULT
                            f_c = sh * oe[j] * ETF_FEE + TL_FEE_YUAN * lots
                            eq -= s_c * slip_on + f_c * fee_on
                            slip_tot += s_c * slip_on
                            fee_tot += f_c * fee_on
                            pe, pf = float(sh), -float(lots)
                elif want == 0 and pe > 0:
                    lots = -pf
                    s_c = pe * ETF_SLIP_TICKS * ETF_TICK + lots * TL_SLIP_TICKS * TL_TICK * TL_MULT
                    f_c = pe * oe[j] * ETF_FEE + TL_FEE_YUAN * lots
                    eq -= s_c * slip_on + f_c * fee_on
                    slip_tot += s_c * slip_on
                    fee_tot += f_c * fee_on
                    pnls.append(eq - entry_eq)
                    pe, pf = 0.0, 0.0
        eq += pe * (ce[j] - oe[j]) + pf * TL_MULT * (cf[j] - of_[j])
        curve[j] = eq
    idx = np.array([j for j in range(bt_i, N) if m['last'][j]])
    return {'nav': curve[idx] / INIT_CAPITAL, 'dates': m['dates'][idx], 'pnls': np.array(pnls),
            'slip': slip_tot, 'fee': fee_tot, 'final': curve[N - 1] - INIT_CAPITAL}


# =============================== 绩效 ===============================
def perf(nav, dates):
    """nav: 日度净值(不含起点1.0)；dates 与 nav 对齐"""
    nv = np.r_[1.0, np.asarray(nav, float)]
    dts = [None] + list(dates)
    n = len(nv) - 1
    if n < 2:
        return None
    r = nv[1:] / nv[:-1] - 1
    ann = nv[-1] ** (252.0 / n) - 1
    peak = np.maximum.accumulate(nv)
    dd = nv / peak - 1
    mdd = dd.min()
    t = int(np.argmin(dd))
    p = int(np.argmax(nv[:t + 1]))
    sd = r.std(ddof=1)
    sharpe = r.mean() / sd * np.sqrt(252) if sd > 0 else np.nan
    calmar = ann / abs(mdd) if mdd < 0 else np.nan
    return {'ann': ann, 'mdd': mdd, 'sharpe': sharpe, 'calmar': calmar,
            'dd_start': dts[p] if dts[p] is not None else '-', 'dd_end': dts[t] if dts[t] is not None else '-'}


def trade_stats(pnls):
    if len(pnls) == 0:
        return np.nan, np.nan
    win = np.mean(pnls > 0)
    w, l = pnls[pnls > 0], pnls[pnls < 0]
    plr = (w.mean() / abs(l.mean())) if (len(w) and len(l)) else np.nan
    return win, plr


def pct(x):
    return '-' if x is None or (isinstance(x, float) and np.isnan(x)) else '%.2f%%' % (x * 100)


def num(x, f='%.2f'):
    return '-' if x is None or (isinstance(x, float) and np.isnan(x)) else f % x


def yearly_table(res):
    nav, dts = res['nav'], res['dates']
    years = sorted(set(d.year for d in dts))
    out = {}
    prev = 1.0
    for y in years:
        mk = np.array([d.year == y for d in dts])
        seg = nav[mk] / prev
        pr = perf(seg, dts[mk])
        prev = nav[mk][-1]
        if pr:
            out[str(y)] = [pct(pr['ann']), pct(pr['mdd']), num(pr['sharpe']), num(pr['calmar']),
                           str(pr['dd_start']), str(pr['dd_end'])]
    return pd.DataFrame(out, index=['年化收益率', '最大回撤', '夏普比率', '卡玛比率', '最大回撤开始时间', '最大回撤结束时间'])


# =============================== 研报上篇：累计收益率之差策略 ===============================
def part1_strategy(m, bt_i, slip_on, fee_on):
    N = m['N']
    Xb = PART1_X * BARS_PER_DAY
    lnE, lnF = np.log(m['ce']), np.log(m['cf'])
    sp = pd.Series((lnE - np.r_[np.full(Xb, np.nan), lnE[:-Xb]]) - (lnF - np.r_[np.full(Xb, np.nan), lnF[:-Xb]]))
    mu = sp.rolling(Xb, min_periods=Xb).mean()
    sd = sp.rolling(Xb, min_periods=Xb).std()
    zz = ((sp - mu) / sd.replace(0, np.nan)).fillna(10.0).values
    ones = np.ones(N)
    cost = cost_array(dict(m, beta=ones), slip_on, fee_on)
    rets = train_eval(zz, np.zeros(N, int), ones, m['gap_e'], m['gap_f'], m['oc_e'], m['oc_f'],
                      m['last'], cost, np.array([-PART1_K]), np.array([-PART1_K]))[:, 0]
    dr = pd.Series(rets[bt_i:]).groupby(m['day_id'][bt_i:]).sum().values
    nav = np.cumprod(1 + dr)
    dts = m['dates'][bt_i:][np.r_[m['first'][bt_i:][1:], True]]
    return perf(nav, dts)


# =============================== 主流程 ===============================
def run_etf(code, name, tl, bt_date, d_warm, d_end):
    print('\n' + '=' * 90)
    print('%s  %s' % (code, name))
    print('=' * 90)
    etf = get_bars30(code, d_warm, d_end)
    if len(etf) == 0:
        print('无数据')
        return
    etf = etf.rename(columns={'open': 'oe', 'close': 'ce'})
    df = etf.join(tl, how='inner')
    df = df.dropna()
    print('30分钟样本数 %d，区间 %s ~ %s' % (len(df), df.index[0], df.index[-1]))
    if len(df) < (INIT_DAYS + TRAIN_DAYS + 5) * BARS_PER_DAY:
        print('样本不足，无法完成建模与滚动训练（可调小 TRAIN_DAYS / INIT_DAYS 或提前 WARMUP_START）')
        return
    m = build_model(df)
    # 研报表2：日频相关性
    dl = np.where(m['last'])[0]
    pe_d, pf_d = m['ce'][dl], m['cf'][dl]
    print('\n[表2 复现] 日频相关性: 价格相关系数 %.3f | 收益率相关系数 %.3f' %
          (np.corrcoef(pe_d, pf_d)[0, 1], np.corrcoef(np.diff(np.log(pe_d)), np.diff(np.log(pf_d)))[0, 1]))
    bt_ok = m['dates'] >= bt_date
    if not bt_ok.any():
        print('回测区间内无数据')
        return
    bt_i = int(np.argmax(bt_ok))

    if RUN_PART1:
        rows = []
        for nm, sl, fe in [('有交易成本', 1, 1), ('无交易成本', 0, 0)]:
            pr = part1_strategy(m, bt_i, sl, fe)
            if pr:
                rows.append([nm, pct(pr['ann']), pct(pr['mdd']), num(pr['sharpe']), num(pr['calmar'])])
        print('\n[上篇 表4 近似复现] 累计收益率之差策略 (X=%d天,K=%.1f; 30分钟频率，等名义本金，全仓)' % (PART1_X, PART1_K))
        print(pd.DataFrame(rows, columns=['', '年化收益率', '最大回撤', '夏普比率', '卡玛比率']).to_string(index=False))

    nodes = build_nodes(m)
    if not nodes:
        print('无可用的滚动节点')
        return
    st_all = np.concatenate([nd['st'][nd['tr_b'] - nd['tr_a']:] for nd in nodes])
    f_all = m['feat'][nodes[0]['tr_b']:nodes[-1]['te_b']]
    print('\n[HMM] 测试区间状态占比: 状态0(平稳) %.1f%% | 状态1(动荡) %.1f%%; 新息(高频)标准差: 状态0 %.5f, 状态1 %.5f' %
          ((st_all == 0).mean() * 100, (st_all == 1).mean() * 100,
           f_all[st_all == 0, 4].std() if (st_all == 0).any() else np.nan,
           f_all[st_all == 1, 4].std() if (st_all == 1).any() else np.nan))

    scen = [('无手续费无滑点', 0, 0), ('无手续费有滑点', 1, 0), ('有手续费有滑点', 1, 1)]
    results = {}
    for nm, sl, fe in scen:
        results[nm] = run_scenario(m, nodes, sl, fe, bt_i)
    main = results['有手续费有滑点']

    # 表1：主结果
    pr = perf(main['nav'], main['dates'])
    win, plr = trade_stats(main['pnls'])
    print('\n[下篇 表1/表4 复现] %s 套利策略表现（有手续费有滑点）' % code)
    print(pd.DataFrame([[pct(pr['ann']), pct(pr['mdd']), num(pr['sharpe']), num(pr['calmar']),
                         num(main['slip'] / 1e4) + '万', num(main['fee'] / 1e4) + '万',
                         num(main['final'] / 1e4) + '万', pct(win), len(main['pnls'])]],
                       columns=['年化收益率', '最大回撤', '夏普比率', '卡玛比率', '滑点支出', '手续费支出', '扣费后盈亏', '胜率', '交易笔数'],
                       index=[code]).to_string())
    # 表2/5：分年度
    print('\n[下篇 表2/表5 复现] 分年度表现')
    print(yearly_table(main).to_string())
    # 表3/6：不同交易成本
    rows = {}
    for nm, _, _ in scen:
        r = results[nm]
        p = perf(r['nav'], r['dates'])
        w, pl = trade_stats(r['pnls'])
        rows[nm] = [pct(p['ann']), pct(p['mdd']), num(p['sharpe']), num(p['calmar']), num(pl), pct(w),
                    str(p['dd_start']), str(p['dd_end'])]
    print('\n[下篇 表3/表6 复现] 不同交易成本下的净值表现')
    print(pd.DataFrame(rows, index=['年化收益率', '最大回撤', '夏普比率', '卡玛比率', '盈亏比', '胜率',
                                    '最大回撤开始时间', '最大回撤结束时间'])[[s[0] for s in scen]].to_string())
    # 滚动参数
    print('\n[滚动参数] 每个更新节点选出的 (P0, P1) 分位数(%) 及训练集夏普；False=训练集不盈利→空仓')
    rows = [[str(c[0]), c[1], c[2], c[3], num(c[4])] for c in main['chosen']]
    print(pd.DataFrame(rows, columns=['节点日期', 'P0(平稳)', 'P1(动荡)', '是否交易', '训练集夏普']).to_string(index=False))

    if SHOW_PLOT:
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(3, 1, figsize=(12, 10))
            for nm, _, _ in scen:
                ax[0].plot(results[nm]['dates'], results[nm]['nav'], label=nm)
            ax[0].legend()
            ax[0].set_title('%s 套利策略净值' % code)
            ax[1].plot(m['beta'])
            ax[1].set_title('卡尔曼对冲比率')
            ax[2].plot(m['v'])
            ax[2].set_title('新息')
            plt.show()
        except Exception as e:
            print('绘图失败:', e)


def main():
    d_warm = _d(WARMUP_START)
    bt_date = _d(START_DATE)
    d_end = min(_d(END_DATE), datetime.date.today())
    print('提取 TL 主力合约30分钟数据 ...')
    tl = build_tl(d_warm, d_end)
    print('TL 样本数 %d，区间 %s ~ %s' % (len(tl), tl.index[0], tl.index[-1]))
    for code, name in ETF_LIST:
        run_etf(code, name, tl, bt_date, d_warm, d_end)


main()


提取 TL 主力合约30分钟数据 ...
TL 样本数 6690，区间 2023-08-01 10:00:00 ~ 2026-10-30 15:15:00

511130.XSHG  博时上证30年期国债ETF
30分钟样本数 2480，区间 2024-03-29 10:00:00 ~ 2026-10-29 15:00:00

[表3 复现] 流动性因子对回归分析的影响(全样本OLS, 30分钟)
方程     R2 ADF检验P值
无流动性因子  0.986   0.121
有流动性因子  0.986   0.091

[卡尔曼] 对冲比率 beta1: 最小 0.8366  最大 1.1177  均值 1.0671 | 新息均值绝对值 0.00306
[小波] Haar 平稳小波能量占比(近似, 研报图1: A4 0.034 / D4 0.052 / D3 0.064 / D2 0.237 / D1 0.614):
   Approx(L4) 0.935 | Detail L4 0.026 | Detail L3 0.017 | Detail L2 0.012 | Detail L1 0.011
[DDIVF] 最优平滑系数 alpha = 0.50

[表2 复现] 日频相关性: 价格相关系数 0.993 | 收益率相关系数 0.963

[上篇 表4 近似复现] 累计收益率之差策略 (X=5天,K=0.1; 30分钟频率，等名义本金，全仓)
年化收益率    最大回撤   夏普比率   卡玛比率
有交易成本  -0.73%  -2.10%  -0.38  -0.34
无交易成本   2.49%  -1.02%   1.29   2.45

[HMM] 测试区间状态占比: 状态0(平稳) 62.2% | 状态1(动荡) 37.8%; 新息(高频)标准差: 状态0 0.00112, 状态1 0.00133

[下篇 表1/表4 复现] 511130.XSHG 套利策略表现（有手续费有滑点）
              年化收益率    最大回撤   夏普比率   卡玛比率   滑点支出  手续费支出   扣费后盈亏      胜率  交易笔数
511130.XSHG  -0.19%  -0.70%  -0.26  -0.26  6.95万  5.03万  -2.